<a href="https://colab.research.google.com/github/Dzux1308/image-classification/blob/main/Computer_Vision_Final_(dzux1308_gmail).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Ô 1: Kết nối Google Drive và giải nén dự án ---
from google.colab import drive
import os
import zipfile
print("--> Kết nối tới Google Drive...")
drive.mount('/content/drive')

# Sửa lại đường dẫn cho đúng với Google Drive của bạn
zip_path = '/content/drive/MyDrive/Colab Notebooks/pythonProject_backup_20250615_093958.zip'
extract_path = '/content/project'

print(f"--> Giải nén dự án từ '{zip_path}'...")
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Di chuyển vào thư mục gốc của dự án
project_root = extract_path  # Thư mục gốc chính là nơi đã giải nén

os.chdir(project_root)

print("--> Dự án đã được giải nén và sẵn sàng tại:")
!pwd

--> Kết nối tới Google Drive...
Mounted at /content/drive
--> Giải nén dự án từ '/content/drive/MyDrive/Colab Notebooks/pythonProject_backup_20250615_093958.zip'...
--> Dự án đã được giải nén và sẵn sàng tại:
/content/project


In [2]:
import torch
import os
import zipfile

# --- BƯỚC 1: ĐỊNH NGHĨA CÁC ĐƯỜNG DẪN CHÍNH XÁC ---

# DÁN ĐƯỜNG DẪN BẠN VỪA SAO CHÉP TỪ CÂY THƯ MỤC VÀO ĐÂY
friend_checkpoints_zip_path = '/content/drive/MyDrive/Colab Notebooks/Friend_Project/checkpoints-20250615T203301Z-1-001.zip' # <-- DÁN ĐƯỜNG DẪN ĐÚNG VÀO ĐÂY
friend_results_zip_path = '/content/drive/MyDrive/Colab Notebooks/Friend_Project/results-20250615T203301Z-1-001.zip'       # <-- DÁN ĐƯỜNG DẪN ĐÚNG VÀO ĐÂY

# Thư mục tạm để giải nén
temp_extract_path = '/content/friend_project_temp'

# Đường dẫn đến dự án của bạn
my_project_checkpoints_dir = '/content/project/checkpoints'
my_project_results_dir = '/content/project/results'

# --- BƯỚC 2: GIẢI NÉN CÁC FILE ZIP ---

print("--- BẮT ĐẦU GIẢI NÉN ---")
# Xóa thư mục tạm cũ nếu có để đảm bảo làm lại từ đầu
!rm -rf {temp_extract_path}
os.makedirs(temp_extract_path, exist_ok=True)

try:
    print(f"Đang cố giải nén: {friend_checkpoints_zip_path}")
    with zipfile.ZipFile(friend_checkpoints_zip_path, 'r') as zip_ref:
        zip_ref.extractall(temp_extract_path)
    print(f"--> Đã giải nén checkpoints vào: {temp_extract_path}")

    print(f"\nĐang cố giải nén: {friend_results_zip_path}")
    with zipfile.ZipFile(friend_results_zip_path, 'r') as zip_ref:
        zip_ref.extractall(temp_extract_path)
    print(f"--> Đã giải nén results vào: {temp_extract_path}")
except FileNotFoundError:
    print("\nLỖI: KHÔNG TÌM THẤY FILE ZIP. Vui lòng thực hiện lại bước sao chép và dán đường dẫn cho chính xác.")
    # Dừng script nếu không giải nén được
    exit()
except Exception as e:
    print(f"\nLỖI KHÁC KHI GIẢI NÉN: {e}")
    exit()

# --- BƯỚC 3: "BIÊN DỊCH" CÁC FILE CHECKPOINT ĐÃ ĐƯỢC GIẢI NÉN ---

extracted_checkpoints_dir = os.path.join(temp_extract_path, 'checkpoints')
extracted_results_dir = os.path.join(temp_extract_path, 'results')

friend_checkpoints_to_compile = [
    'ConvNeXt_Tiny_Baseline',
    'ConvNeXt_Tiny_RE'
]

print("\n--- BẮT ĐẦU QUÁ TRÌNH 'BIÊN DỊCH' CHECKPOINT ---")

for exp_name in friend_checkpoints_to_compile:
    original_path = os.path.join(extracted_checkpoints_dir, f"{exp_name}.pth")

    print(f"\nĐang xử lý file: {original_path}")

    if not os.path.exists(original_path):
        print("--> File không tồn tại, bỏ qua.")
        continue

    original_checkpoint = torch.load(original_path, map_location='cpu')

    if isinstance(original_checkpoint, dict) and 'net' in original_checkpoint:
         print("--> File này đã có cấu trúc đúng. Không cần biên dịch.")
         continue

    if isinstance(original_checkpoint, dict):
        state_dict = original_checkpoint.get('model_state_dict', original_checkpoint.get('state_dict', original_checkpoint))
    else:
        state_dict = original_checkpoint

    acc = 0.0
    acc_path = os.path.join(extracted_results_dir, f"{exp_name}_acc.txt")
    if os.path.exists(acc_path):
        with open(acc_path, 'r') as f:
            acc = float(f.read().strip())

    new_checkpoint = {
        'net': state_dict,
        'acc': acc,
        'epoch': 99
    }

    torch.save(new_checkpoint, original_path)
    print(f"--> Đã 'biên dịch' và lưu lại file thành công!")

print("\n--- HOÀN TẤT BIÊN DỊCH ---")


# --- BƯỚC 4: GỘP CÁC FILE ĐÃ BIÊN DỊCH VÀO DỰ ÁN CỦA BẠN ---

print("\n--- BẮT ĐẦU GỘP DỮ LIỆU VÀO DỰ ÁN ---")

print("\n--> Đang sao chép các file Checkpoint đã biên dịch...")
!cp -v "{extracted_checkpoints_dir}"/* "{my_project_checkpoints_dir}/"
print("    Sao chép Checkpoints hoàn tất!")

print("\n--> Đang sao chép các file Results...")
!cp -v "{extracted_results_dir}"/* "{my_project_results_dir}/"
print("    Sao chép Results hoàn tất!")

print("\n--- HOÀN TẤT TOÀN BỘ QUÁ TRÌNH! ---")

--- BẮT ĐẦU GIẢI NÉN ---
Đang cố giải nén: /content/drive/MyDrive/Colab Notebooks/Friend_Project/checkpoints-20250615T203301Z-1-001.zip
--> Đã giải nén checkpoints vào: /content/friend_project_temp

Đang cố giải nén: /content/drive/MyDrive/Colab Notebooks/Friend_Project/results-20250615T203301Z-1-001.zip
--> Đã giải nén results vào: /content/friend_project_temp

--- BẮT ĐẦU QUÁ TRÌNH 'BIÊN DỊCH' CHECKPOINT ---

Đang xử lý file: /content/friend_project_temp/checkpoints/ConvNeXt_Tiny_Baseline.pth
--> Đã 'biên dịch' và lưu lại file thành công!

Đang xử lý file: /content/friend_project_temp/checkpoints/ConvNeXt_Tiny_RE.pth
--> Đã 'biên dịch' và lưu lại file thành công!

--- HOÀN TẤT BIÊN DỊCH ---

--- BẮT ĐẦU GỘP DỮ LIỆU VÀO DỰ ÁN ---

--> Đang sao chép các file Checkpoint đã biên dịch...
'/content/friend_project_temp/checkpoints/ConvNeXt_Tiny_Baseline.pth' -> '/content/project/checkpoints/ConvNeXt_Tiny_Baseline.pth'
'/content/friend_project_temp/checkpoints/ConvNeXt_Tiny_RE.pth' -> '/cont

In [3]:
# --- Ô 2: Cài đặt môi trường và tải dữ liệu ---
print("\n--> Cài đặt các thư viện từ requirements.txt...")
print("\n--> Thiết lập xác thực Kaggle...")
kaggle_config_dir = '/root/.kaggle'
os.makedirs(kaggle_config_dir, exist_ok=True)
# Sửa lại đường dẫn cho đúng với file kaggle.json trên Drive của bạn
kaggle_json_path_drive = '/content/drive/MyDrive/Colab Notebooks/kaggle.json'
!cp "{kaggle_json_path_drive}" "{kaggle_config_dir}/"
!chmod 600 "{kaggle_config_dir}/kaggle.json"

print("\n--> Kiểm tra và tải bộ dữ liệu từ Kaggle...")
if not os.path.exists('data/garbage_classification'):
    print("--> Thư mục dữ liệu chưa có, bắt đầu tải...")
    !kaggle datasets download -d asdasdasasdas/garbage-classification -p data --unzip
else:
    print("--> Thư mục dữ liệu đã tồn tại, bỏ qua bước tải lại.")

print("\n✅✅✅ Môi trường đã sẵn sàng để huấn luyện!")


--> Cài đặt các thư viện từ requirements.txt...

--> Thiết lập xác thực Kaggle...

--> Kiểm tra và tải bộ dữ liệu từ Kaggle...
--> Thư mục dữ liệu chưa có, bắt đầu tải...
Dataset URL: https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification
License(s): copyright-authors
  0% 0.00/82.0M [00:00<?, ?B/s]
100% 82.0M/82.0M [00:00<00:00, 1.54GB/s]

✅✅✅ Môi trường đã sẵn sàng để huấn luyện!


In [ ]:
!python /content/project/run_experiments.py

\n==================== CHECKING EXPERIMENT 1/16: ResNet18_Baseline ====================
--> Found result file. Skipping.
\n==================== CHECKING EXPERIMENT 2/16: ResNet18_RE ====================
--> Found result file. Skipping.
\n==================== CHECKING EXPERIMENT 3/16: ResNet18_SE ====================
--> Found result file. Skipping.
\n==================== CHECKING EXPERIMENT 4/16: ResNet18_SE_RE ====================
--> Found result file. Skipping.
\n==================== CHECKING EXPERIMENT 5/16: MobileNetV2_Baseline ====================
--> Found result file. Skipping.
\n==================== CHECKING EXPERIMENT 6/16: MobileNetV2_RE ====================
--> Found result file. Skipping.
\n==================== CHECKING EXPERIMENT 7/16: MobileNetV2_SE ====================
--> Found result file. Skipping.
\n==================== CHECKING EXPERIMENT 8/16: MobileNetV2_SE_RE ====================
--> Found result file. Skipping.
\n==================== CHECKING EXPERIMENT 9/16: V

In [ ]:
# --- Ô 4: Tổng hợp, Hiển thị và Tải kết quả (Đã sửa lỗi) ---
import pandas as pd
from google.colab import files
from IPython.display import display
import os  # <--- LỖI 1: Thêm dòng này để import thư viện os

# --- LỖI 2: Sửa lại đường dẫn cho chính xác ---
# Script của chúng ta lưu file với tên 'project_summary_final.xlsx'
# và nó nằm trong thư mục /content/project/results/
results_path = '/content/project/results/project_summary_final.xlsx'

# Bây giờ code sẽ chạy đúng
if os.path.exists(results_path):
    print("\n--> Bảng tổng kết kết quả cuối cùng:")
    df = pd.read_excel(results_path)
    display(df) # Dùng display() để hiển thị DataFrame đẹp hơn trong Colab

    print("\n--> Tải file Excel kết quả về máy:")
    files.download(results_path)
else:
    print(f"Lỗi: Không tìm thấy file tại '{results_path}'.")
    print("Quá trình huấn luyện có thể đã gặp lỗi hoặc chưa chạy đến bước tổng kết.")

import os

# 1. Định nghĩa đường dẫn mà Streamlit đang tìm kiếm
target_dir = '/content/project/results'
save_path = os.path.join(target_dir, 'project_summary_with_SOTA.xlsx')

# 2. Tạo thư mục 'results' nếu nó chưa tồn tại
if not os.path.exists(target_dir):
    os.makedirs(target_dir)
    print(f"--- Đã tạo thư mục: {target_dir}")

# 3. Lưu DataFrame (giả sử tên biến bảng của bạn là results_df hoặc df_final)
# Bạn hãy kiểm tra tên biến chứa cái bảng ở trên là gì để thay vào cho đúng nhé
try:
    # Ở đây tôi đoán tên biến là results_df dựa trên ngữ cảnh
    results_df.to_excel(save_path, index=False)
    print(f"--- Đã lưu file thành công tại: {save_path}")
except NameError:
    # Nếu báo lỗi NameError, hãy thay 'results_df' bằng tên biến bảng của bạn
    print("Lỗi: Hãy kiểm tra lại tên biến chứa bảng kết quả của bạn.")

# 4. Kiểm tra xem file đã xuất hiện chưa
if os.path.exists(save_path):
    print("✅ File đã sẵn sàng cho Streamlit!")


--> Bảng tổng kết kết quả cuối cùng:


,Experiment Name,Base Model,SE-Block,Random Erasing,Best Validation Accuracy (%)
0,ResNet18_Baseline,ResNet18,False,False,73.122530
1,ResNet18_RE,ResNet18,False,True,79.249012
2,ResNet18_SE,ResNet18,True,False,74.308300
3,ResNet18_SE_RE,ResNet18,True,True,77.075100
4,MobileNetV2_Baseline,mobilenet_v2,False,False,92.490119
5,MobileNetV2_RE,mobilenet_v2,False,True,93.280632
6,MobileNetV2_SE,mobilenet_v2,True,False,91.106700
7,MobileNetV2_SE_RE,mobilenet_v2,True,True,93.083000
8,VGG16_Baseline,vgg16,False,False,92.292490
9,VGG16_RE,vgg16,False,True,89.723300



--> Tải file Excel kết quả về máy:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Lỗi: Hãy kiểm tra lại tên biến chứa bảng kết quả của bạn.


In [ ]:
import os
import pandas as pd

# 1. Định nghĩa đường dẫn
target_dir = '/content/project/results'
save_path = os.path.join(target_dir, 'project_summary_with_SOTA.xlsx')

if not os.path.exists(target_dir):
    os.makedirs(target_dir)

# 2. Tự động tìm biến DataFrame trong bộ nhớ
found_df = None

# Thử các tên biến phổ biến thường dùng trong dự án này
for name in ['results_df', 'df', 'final_df', 'summary_df', 'df_results']:
    if name in locals():
        found_df = locals()[name]
        print(f"--- Đã tìm thấy bảng với tên biến là: {name}")
        break

# 3. Nếu vẫn không thấy, tìm bất kỳ biến nào là DataFrame
if found_df is None:
    for var_name in list(locals().keys()):
        if isinstance(locals()[var_name], pd.DataFrame):
            # Bỏ qua các biến hệ thống của pandas
            if not var_name.startswith('_'):
                found_df = locals()[var_name]
                print(f"--- Đã tự động nhận diện bảng: {var_name}")
                break

# 4. Lưu file
if found_df is not None:
    found_df.to_excel(save_path, index=False)
    print(f"✅ Đã lưu file thành công tại: {save_path}")
    print("Bây giờ bạn có thể quay lại trang Streamlit và nhấn F5!")
else:
    print("❌ Vẫn không tìm thấy bảng dữ liệu nào. Bạn hãy chạy lại ô code tạo bảng kết quả ở trên một lần nữa rồi chạy lại ô này nhé.")

--- Đã tìm thấy bảng với tên biến là: df
✅ Đã lưu file thành công tại: /content/project/results/project_summary_with_SOTA.xlsx
Bây giờ bạn có thể quay lại trang Streamlit và nhấn F5!


In [ ]:
# --- Ô 5: TẠO SƠ ĐỒ PHÂN TÍCH (Đã sửa lỗi) ---
import pandas as pd
import plotly.express as px
import numpy as np
import os

# --- PHẦN ĐÃ SỬA LỖI ---
# Đường dẫn đúng đến file kết quả cuối cùng
results_path = '/content/project/results/project_summary_final.xlsx'

# Kiểm tra xem file có tồn tại không trước khi vẽ
if os.path.exists(results_path):
    df = pd.read_excel(results_path)

    # --- Bước 1: Làm sạch dữ liệu ---
    # Bỏ các dòng có kết quả 0.0 (là các thử nghiệm chưa chạy xong)
    df_cleaned = df[df['Best Validation Accuracy (%)'] > 0].copy()

    # --- Bước 2: Tạo cột Nhóm để dễ vẽ biểu đồ ---
    # Tạo một cột mới để phân loại các phương pháp cải tiến
    conditions = [
        (df_cleaned['SE-Block'] == True) & (df_cleaned['Random Erasing'] == True),
        (df_cleaned['SE-Block'] == True),
        (df_cleaned['Random Erasing'] == True)
    ]
    choices = ['SE + RE', 'Chỉ SE-Block', 'Chỉ Random Erasing']
    df_cleaned['Loại cải tiến'] = np.select(conditions, choices, default='Baseline (Không cải tiến)')

    # --- Bước 3: Vẽ biểu đồ ---
    fig = px.bar(
        df_cleaned,
        x='Base Model',  # Nhóm theo mô hình cơ sở
        y='Best Validation Accuracy (%)',
        color='Loại cải tiến',  # Tạo các cột màu khác nhau cho mỗi loại cải tiến
        barmode='group',  # Đặt các cột cạnh nhau thay vì chồng lên nhau
        text_auto='.2f',  # Hiển thị giá trị trên mỗi cột, làm tròn 2 chữ số
        title="<b>Biểu đồ So sánh Hiệu quả Cải tiến trên Từng Mô hình</b>",
        labels={
            'Base Model': 'Mô hình Cơ sở',
            'Best Validation Accuracy (%)': 'Độ chính xác Kiểm định (%)',
            'Loại cải tiến': 'Phương pháp Cải tiến'
        }
    )

    # Tinh chỉnh giao diện cho dễ nhìn hơn
    fig.update_traces(textangle=0, textposition="outside")
    fig.update_layout(
        yaxis_title="Độ chính xác Kiểm định (%)",
        xaxis_title="Mô hình Cơ sở",
        legend_title_text="<b>Phương pháp Cải tiến</b>",
        font=dict(size=12),
        # Phóng to sự khác biệt bằng cách bắt đầu trục Y từ 70%
        yaxis_range=[70, df_cleaned['Best Validation Accuracy (%)'].max() + 2]
    )

    fig.show()

else:
    print(f"Lỗi: Không tìm thấy file '{results_path}'.")
    print("Vui lòng chạy lại ô tổng hợp kết quả trước.")

In [ ]:
# --- Ô 6 (PHIÊN BẢN TỐI ƯU): LƯU TIẾN TRÌNH RA GOOGLE DRIVE ---
import shutil
import datetime
import os

print("--> Bắt đầu quá trình backup thông minh (bỏ qua thư mục data)...")

# --- CÁC ĐƯỜNG DẪN VÀ THIẾT LẬP ---
project_path = '/content/project'
backup_temp_dir = '/content/backup_temp' # Thư mục tạm để chứa những thứ cần backup
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
archive_name = f'pythonProject_backup_{timestamp}'
destination_path_drive = f'/content/drive/MyDrive/Colab Notebooks/{archive_name}'

# --- DANH SÁCH NHỮNG THỨ CẦN BACKUP ---
# Chúng ta chỉ backup những thư mục và file quan trọng, có thay đổi
items_to_backup = [
    'models',
    'results',
    'checkpoints',
    'app.py',
    'train.py',
    'run_experiments.py'
    # Lưu ý: Thư mục 'data' đã được cố tình bỏ qua!
]

# --- BẮT ĐẦU QUÁ TRÌNH BACKUP ---
try:
    # 1. Tạo một thư mục tạm "sạch sẽ"
    if os.path.exists(backup_temp_dir):
        shutil.rmtree(backup_temp_dir)
    os.makedirs(backup_temp_dir)
    print(f"--> Đã tạo thư mục tạm tại: {backup_temp_dir}")

    # 2. Sao chép những thứ cần thiết vào thư mục tạm
    print("--> Bắt đầu sao chép các file và thư mục quan trọng...")
    for item in items_to_backup:
        source_item_path = os.path.join(project_path, item)
        dest_item_path = os.path.join(backup_temp_dir, item)
        if os.path.exists(source_item_path):
            if os.path.isdir(source_item_path):
                shutil.copytree(source_item_path, dest_item_path)
            else:
                shutil.copy2(source_item_path, dest_item_path)
            print(f"    - Đã sao chép: {item}")
        else:
            print(f"    - Cảnh báo: Không tìm thấy '{item}', bỏ qua.")

    # 3. Nén thư mục tạm này lại (giờ nó đã nhẹ hơn rất nhiều)
    print(f"--> Đang nén thư mục tạm...")
    shutil.make_archive(destination_path_drive, 'zip', backup_temp_dir)

    print(f"\n✅ ĐÃ LƯU TIẾN TRÌNH THÀNH CÔNG!")
    print(f"File mới đã được tạo tại Google Drive: {destination_path_drive}.zip")

finally:
    # 4. Dọn dẹp thư mục tạm sau khi hoàn tất
    if os.path.exists(backup_temp_dir):
        shutil.rmtree(backup_temp_dir)
        print("\n--> Đã dọn dẹp thư mục tạm.")

In [4]:
!pip install -q streamlit pyngrok pandas plotly torch torchvision openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 113.5 MB/s eta 0:00:00


In [5]:
from pyngrok import ngrok
import os

# Kill các tunnel cũ nếu có
ngrok.kill()

# Thay 'YOUR_NGROK_AUTHTOKEN' bằng token của bạn
NGROK_AUTH_TOKEN = "2yP7W56daa0neOGrDOzJRI8GziG_7LKKFowA6rgbEr7JL6E8S" # <--- THAY TOKEN CỦA BẠN VÀO ĐÂY
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Mở một tunnel tới port 8501
public_url = ngrok.connect(8501)
print(f"Link MỚI để truy cập ứng dụng Streamlit của bạn: {public_url}")

# Chạy streamlit trong nền
!streamlit run /content/project/app.py --server.runOnSave true &

Link MỚI để truy cập ứng dụng Streamlit của bạn: NgrokTunnel: "https://1b9f-136-117-30-209.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.117.30.209:8501

2026-02-14 17:27:29.943 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
  Stopping...


In [ ]:
import os
import pandas as pd

# 1. Đường dẫn file
target_dir = '/content/project/results'
save_path = os.path.join(target_dir, 'project_summary_with_SOTA.xlsx')

# 2. Tìm lại biến bảng dữ liệu (như bước trước)
found_df = None
for name in ['results_df', 'df', 'final_df', 'summary_df', 'df_results']:
    if name in locals():
        found_df = locals()[name].copy() # Tạo bản sao để tránh báo lỗi SettingWithCopy
        break

if found_df is not None:
    # 3. ĐỔI TÊN CỘT SANG TIẾNG VIỆT ĐỂ KHỚP VỚI STREAMLIT
    mapping = {
        'Experiment Name': 'Tên Thử nghiệm',
        'Best Validation Accuracy (%)': 'Độ chính xác Tốt nhất (%)',
        'Base Model': 'Mô hình Cơ sở'
    }
    found_df = found_df.rename(columns=mapping)

    # 4. Lưu đè lên file cũ
    found_df.to_excel(save_path, index=False)
    print(f"✅ Đã cập nhật tên cột và lưu tại: {save_path}")
    print("Bây giờ hãy quay lại trang Streamlit và F5 (Refresh) lại nhé!")
else:
    print("❌ Không tìm thấy biến bảng dữ liệu. Hãy chạy lại ô code tạo bảng kết quả của bạn trước.")

✅ Đã cập nhật tên cột và lưu tại: /content/project/results/project_summary_with_SOTA.xlsx
Bây giờ hãy quay lại trang Streamlit và F5 (Refresh) lại nhé!
